# Metabolic Engineering and Flux Balance Analysis
In this lecture, we will introduce flux balance analysis (FBA), a tool based on linear programming, to estimate reaction rates (called fluxes) in a metabolic network. To solve the FBA problem, we will construct the stoichiometric matrix, formulate possible bounds on the unknown fluxes, and use reaction reversibility and enzyme capacity to set the flux bounds. 

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Represent a metabolic reaction network as a stoichiometric matrix:__ The stoichiometric matrix is the digital representation of the reaction biochemistry occurring inside a cell. We'll construct a stoichiometric matrix from biochemical reactions and interpret its rows, columns, and coefficient signs.
> * __Formulate a flux balance analysis problem:__ The flux balance analysis problem is a linear programming problem that can be solved to estimate the unknown fluxes in a metabolic network. We'll formulate the FBA problem, explain the objective function, and discuss the constraints that define the feasible solution space.
> * __Develop and interpret flux bounds:__ The flux bounds are constraints on the unknown fluxes that define the feasible solution space. We'll discuss how to set flux bounds based on reaction reversibility, enzyme capacity, and interpret the biological meaning of these bounds.

We'll demonstrate these concepts with an example, where we estimate the metabolic fluxes in the urea cycle of a mammalian liver cell. Let's get started!
___

## Examples
We will use the following example to connect the flux balance formulation to a numerical calculation:

> [▶ Calculate fluxes in a urea-cycle model](CHEME-5800-L6a-Example-UreaCycle-FluxBalance-Fall-2026.ipynb). In this example, we will build a stoichiometric model of the urea cycle in Human cells, estimate reaction directions and enzyme capacities to establish the flux bounds, and solve a linear program to maximize urea export. While this example is specific to the urea cycle, the same approach can be applied to any metabolic network. Further, it demonstrates the integrative nature of flux balance analysis. 

The [companion derivation notebook](CHEME-5800-L6a-Advanced-Derivation-FluxBalanceAnalysis-Fall-2026.ipynb) develops the steady-state material balances used in the lecture from a mole balance.
___

## Metabolic Engineering
Metabolic pathways encode the enzyme catalyzed reactions that convert nutrients into biomass and energy in living cells. There is an amazing diversity of metabolic pathways in different organisms, but also some highly conserved pathways that are shared across many species, such as __central metabolism__ (glycolysis, the pentose phosphate pathway, and the TCA cycle).  We can explore these connections in the [KEGG metabolic pathways map (map01100)](https://www.kegg.jp/pathway/map01100).

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<img class="course-diagram" src="figs/Fig-Central-Metabolism/Fig-Central-Metabolism.svg" width="1100" alt="Selected routes through glycolysis, the pentose phosphate pathway, and the TCA cycle, with branches toward lactate, nucleotides, lipids, and proteins. ATP and redox-carrier yields accompany the pathways, with a circular TCA cycle and an oxidative phosphorylation inset. Arrows may combine multiple reactions.">

These are the reaction networks that we need to manipulate in order to produce a desired compound. The systematic study and analysis of these networks is called __metabolic engineering__. 

> __What is metabolic engineering?__
>
> Metabolic engineering is the directed modification of an organism's metabolic pathways through genetic and regulatory changes to increase the production of a desired compound or to enable the synthesis of new products. The engineering question is: how can we redirect the flow of carbon, nitrogen, and energy through the network toward a target molecule while meeting the cell's other requirements?
> 
> Two landmark papers that defined the field appeared in the same 1991 issue of _Science_:
> * [Bailey JE. Toward a science of metabolic engineering. Science. 1991 Jun 21;252(5013):1668–75.](https://pubmed.ncbi.nlm.nih.gov/2047876/)
> * [Stephanopoulos G, Vallino JJ. Network rigidity and metabolic engineering in metabolite overproduction. Science. 1991 Jun 21;252(5013):1675–81.](https://pubmed.ncbi.nlm.nih.gov/1904627/)

We describe the flow of material through the network using __metabolic fluxes__: reaction rates expressed per unit volume or biomass. Which combinations of reaction rates can sustain production of a desired compound, and what limits the production rate? 

Flux balance analysis (FBA) is a quantitative tool that provides a way to answer these questions.

___

## Flux Balance Analysis
__Flux balance analysis (FBA)__ estimates (optimal) metabolic reaction rates (fluxes) from material balances and biological constraints. FBA uses linear programming to find the flux distribution (reaction rates) that optimizes a chosen objective function, subject to stoichiometric and capacity constraints, at a __pseudo-steady state__.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<p align="center">
<img class="course-diagram" src="figs/Fig-FBA-FeasibleSpace/Fig-FBA-FeasibleSpace.svg" width="1100" style="max-width:100%; height:auto;" alt="Three panels show steady-state flux directions in the null space, bounds restricting them to a feasible polytope, and a linear objective reaching an optimal edge. Multiple points on the edge are equally optimal. Projecting the edge onto flux v1 gives its range at the optimal objective value.">
</p>

__Conservation, bounds, and alternate optima.__ The balance equations define the null space $\mathcal N(\mathbf S)$; bounds restrict it to the feasible set $\mathcal P$. The objective contours move toward larger values until they reach the highlighted edge. Every point on this edge has the same optimal objective value $Z^\star$, so the optimal flux distribution is not unique. __Flux variability analysis (FVA)__ fixes the objective at $Z^\star$ and finds the minimum and maximum of each reaction flux; the red bracket shows this interval for $\hat v_1$.

Adapted from the [Varner lab chapter's linear-program geometry figure](https://github.com/varnerlab/MRW-BTC4-Chapter-Varner/blob/881056b3c3c57d83036f2c219d81c848d763118d/chapter/figures/lp_geometry_fig/lp_geometry.tex). Let's look at the different components of the FBA problem and how they are formulated, starting with the stoichiometric matrix.

<!-- In the urea example, urea leaves through an exchange with a negative flux, so we maximize urea export by setting its objective coefficient to $-1$ and all other coefficients to zero. For more on the formulation, see [Orth et al. (2010)](https://pmc.ncbi.nlm.nih.gov/articles/PMC3108565/) and [Heirendt et al. (2019)](https://pubmed.ncbi.nlm.nih.gov/30787451/). Next, we will use enzyme capacity and reaction direction to specify the flux bounds. -->

### Stoichiometric Matrix
We formulate the stoichiometric matrix $\mathbf{S}$ from a __metabolic reconstruction__. A metabolic reconstruction describes the biochemical processes operating inside a cell, using a curated set of reactions, metabolites, and stoichiometric coefficients.  We can obtain metabolic reconstructions and supporting biological data from several resources:

| Resource | What we use it for |
|:---|:---|
| [KEGG (Kanehisa et al., 2025)](https://academic.oup.com/nar/article/53/D1/D672/7824602) | Explore pathways and relationships among genes, enzymes, and reactions. |
| [BioCyc (Karp et al., 2019)](https://pubmed.ncbi.nlm.nih.gov/29447345/) | Examine organism-specific pathway and genome information. |
| [BiGG Models (Norsigian et al., 2020)](https://academic.oup.com/nar/article/48/D1/D402/5614178) | Obtain genome-scale metabolic models with standardized metabolite and reaction identifiers. |
| [BRENDA (Chang et al., 2021)](https://academic.oup.com/nar/article/49/D1/D498/5992283) | Find enzyme properties and kinetic measurements that can inform flux bounds. |
| [BioNumbers (Milo et al., 2010)](https://academic.oup.com/nar/article/38/suppl_1/D750/3112244) | Find measured biological quantities and their literature sources. |


The stoichiometric matrix $\mathbf{S}$ encodes the reaction network structure, which is the set of reactions and their stoichiometric coefficients. The columns of the stoichiometric matrix correspond to reactions, and the rows correspond to chemical species (metabolites).

> __Definition: Stoichiometric matrix__
>
> Let $\mathcal{M}$ denote the set of modeled chemical species and $\mathcal{R}$ the set of reactions. The stoichiometric matrix $\mathbf{S}$ is given by:
>
> $$
> \mathbf{S}=[\sigma_{ij}]
> \in\mathbb{R}^{|\mathcal{M}|\times|\mathcal{R}|}.
> $$
>
> Here, $|\mathcal{M}|$ is the number of modeled species and $|\mathcal{R}|$ is the number of reactions. The entry $\sigma_{ij}$ is the net stoichiometric coefficient of species $i$ in reaction $j$, with signs defined relative to the reaction's written forward direction:
>
> * $\sigma_{ij}>0$: species $i$ is __produced__ by the reaction $j$, i.e., species $i$ is a __product__ of reaction $j$.
> * $\sigma_{ij}<0$: species $i$ is __consumed__ by the reaction $j$, i.e., species $i$ is a __reactant__ of reaction $j$.
> * $\sigma_{ij}=0$: species $i$ is not involved in reaction $j$ (or this is a net zero stoichiometry).
>
> A zero entry can mean that the species is absent from the reaction or that its reactant and product coefficients cancel. The matrix records net stoichiometry. However, it does not encode regulatory or control mechanisms; regulation enters through the flux bounds, not the stoichiometric matrix.

#### Example: Stoichiometric Column
For example, consider a reaction $j$ that consumes one unit of $A$ and two units of $B$ to produce one unit of $C$.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<p align="center">
<img class="course-diagram" src="figs/Fig-Stoichiometric-ControlVolume/Fig-Stoichiometric-ControlVolume.svg" width="780" style="max-width:100%; height:auto;" alt="A plus two B react to form C inside a dashed control-volume boundary. Material enters from the surroundings on the left and leaves on the right; empty-set symbols denote the surroundings.">
</p>

The dashed boundary encloses the modeled species. Material can enter or leave the control volume through the boundary. With the species ordered as $(A,B,C)$, the $j$-th column of the stoichiometric matrix is given by:

$$
A+2B\longrightarrow C,
\qquad
\mathbf{s}_j=
\begin{bmatrix}
-1\\
-2\\
1
\end{bmatrix}.
$$

Multiplying the stoichiometric column by the reaction flux gives the reaction's contribution to the three species balances. The coefficient $-2$ means that the consumption rate of $B$ is twice the reaction flux. For a reversible reaction with a negative flux, the signs of these contributions reverse; we keep the same stoichiometric column.

An __exchange reaction__ connects a metabolite to the surroundings, written as $\emptyset$ (which we do not balance). Its column in the stoichiometric matrix has a single nonzero entry, e.g., for the reactant $A$ being produced by exchange reaction $k$, the corresponding column of the stoichiometric matrix is given by:

$$
\emptyset\longrightarrow A,
\qquad
\mathbf{s}_k=
\begin{bmatrix}
1\\
0\\
0
\end{bmatrix}.
$$

A positive flux brings $A$ into the control volume, and a negative flux removes it. The urea example writes every exchange this way. However, the BiGG models that we explore [in the L6b lab notebook](../L6b/CHEME-5800-L6b-Lab-OverflowMetabolism-Fall-2026.ipynb) write $A\longrightarrow\emptyset$, so uptake is negative there (different conventions are used in the literature). The sign convention does not affect the FBA solution, but it does affect the interpretation of the fluxes.

### A Model for Flux Bounds
The flux bounds are important constraints in flux balance analysis calculations and the convex decomposition of the stoichiometric array. Beyond their role in the flux estimation problem, the flux bounds are _integrative_, i.e., these constraints integrate many types of genetic and biochemical information into the problem. A general model for these bounds is given by:

$$
\underbrace{
-\delta_j\overbrace{
\left[V_{max,j}^{\circ}\left(\frac{e}{e^{\circ}}\right)\theta_j(\dots)f_j(\dots)\right]
}^{\text{assumed reverse capacity}}
}_{\mathcal{L}_j}
\leq\hat v_j\leq
\underbrace{
V_{max,j}^{\circ}\left(\frac{e}{e^{\circ}}\right)\theta_j(\dots)f_j(\dots)
}_{\mathcal{U}_j}.
$$
where $V_{max,j}^{\circ}$ denotes the maximum reaction velocity (units: `flux`) computed at some _characteristic enzyme abundance_. Thus, the maximum reaction velocity is given by:
$$
V_{max,j}^{\circ} = k_{cat,j}^{\circ}e^{\circ}
$$
where $k_{cat,j}$ is the catalytic constant or turnover number for the enzyme (units: `1/time`) and $e^{\circ}$ is a characteristic enzyme abundance (units: `concentration`). The term $\left(e/e^{\circ}\right)$ is a correction to account for the _actual_ enzyme abundance catalyzing the reaction (units: `dimensionless`).  

The following table defines the model quantities. We express enzyme abundance and reaction flux per gram of cell dry weight:

<table style="width:100%;table-layout:fixed;text-align:left;">
<colgroup><col style="width:23%;"><col style="width:77%;"></colgroup>
<thead><tr><th style="text-align:left;">Quantity</th><th style="text-align:left;">Meaning</th></tr></thead>
<tbody>
<tr><td><i>V</i><sub>max,j</sub><sup>∘</sup></td><td>Maximum reaction velocity at the characteristic enzyme abundance (mmol gDW<sup>−1</sup> h<sup>−1</sup>).</td></tr>
<tr><td><i>k</i><sub>cat,j</sub><sup>∘</sup></td><td>Reference catalytic turnover number (h<sup>−1</sup>).</td></tr>
<tr><td><i>e</i><sup>∘</sup> and <i>e</i></td><td>Characteristic and actual abundances of the enzyme that catalyzes reaction <i>j</i> (mmol gDW<sup>−1</sup>).</td></tr>
<tr><td><i>θ</i><sub>j</sub>(…) ∈ [0, 1]</td><td>Fraction of maximal enzyme activity that remains under allosteric regulation; (…) stands for the concentrations of the enzyme's activators and inhibitors.</td></tr>
<tr><td><i>f</i><sub>j</sub>(…) ∈ [0, 1]</td><td>Substrate saturation; (…) stands for the substrate concentrations. Equal to one when the substrates are saturating.</td></tr>
<tr><td><i>δ</i><sub>j</sub> ∈ {0, 1}</td><td>Zero for a forward-only reaction; one for a reversible reaction.</td></tr>
</tbody>
</table>

The bounds model assumes equal forward and reverse capacities for reversible reactions; different capacities would need separate parameters for the reverse direction. 

#### Simplified bounds model
Let's initially assume that $(e/e^{\circ})\sim{1}$, there are no allosteric inputs $\theta_{j}\left(\dots\right)\sim{1}$, and the substrates are saturating $f_{j}\left(\dots\right)\sim{1}$. 
Then, the flux bounds are given by:
$$
\begin{align*}
\underbrace{-\delta_{j}V_{max,j}^{\circ}}_{\mathcal{L}_j}\leq\hat v_j\leq\underbrace{V_{max,j}^{\circ}}_{\mathcal{U}_j}
\end{align*}
$$
This is a simple model for the flux bounds. It is easy to see that the flux bounds are a function of the maximum reaction velocity, the catalytic constant or turnover number, and our assumed value of a characteristic enzyme abundance.

### Linear Programming Formulation
The flux balance analysis problem has the same structure as the minimum-cost flow problem we explored previously: the stoichiometric matrix $\mathbf{S}$ plays the role of the incidence matrix $\mathbf{A}$, with the same signs, and the exchange reactions take over the role of the source and sink terms, so the right-hand side is zero. Unlike a column of the incidence matrix, a reaction column can have more than two nonzero entries and coefficients other than $\pm1$.

The system of material balances that form the linear system of constraints rest on two assumptions, which the [companion derivation notebook](CHEME-5800-L6a-Advanced-Derivation-FluxBalanceAnalysis-Fall-2026.ipynb) develops from a mole balance. The amount of each metabolite per gram of biomass stays constant, and the extra production needed to offset dilution by new biomass, called growth dilution, is small compared with each metabolite's production and consumption rates. 

Under these assumptions, the __flux balance analysis problem__ is given by:

> __Flux balance analysis (FBA) problem__
>
> Let $\mathcal{M}$ denote the set of modeled species and $\mathcal{R}$ the set of reactions, including exchange reactions. The coefficient $\sigma_{ij}$, the $(i,j)$ entry of $\mathbf{S}$, is the net stoichiometric coefficient of species $i\in\mathcal{M}$ in reaction $j\in\mathcal{R}$.
>
> For each reaction $j$, $\hat v_j$ is the unknown flux, $c_j$ is its weight in the objective, and $\mathcal{L}_j$ and $\mathcal{U}_j$ are its lower and upper bounds. The vector $\hat{\mathbf{v}}$ collects the fluxes. Fluxes and bounds are expressed in $\mathrm{mmol\,gDW^{-1}\,h^{-1}}$, millimoles per gram of cell dry weight per hour. We fix the stoichiometric coefficients, objective weights, and flux bounds before solving the following problem:
>
> $$
> \boxed{
> \begin{aligned}
> \underset{\hat{v}_1,\hat{v}_2,\ldots,\hat{v}_{|\mathcal{R}|}}{\operatorname{maximize}}\quad
> & \sum_{j\in\mathcal{R}}c_j\hat v_j \\
> \text{subject to}\quad
> & \sum_{j\in\mathcal{R}}\sigma_{ij}\hat v_j=0,
> && i\in\mathcal{M},\quad\text{(material balance constraints)}\\
> & \mathcal{L}_j\leq\hat v_j\leq\mathcal{U}_j,
> && j\in\mathcal{R}\quad\text{(thermodynamics and kinetic constraints)}.
> \end{aligned}}
> $$
>
> In matrix form, the balance constraints are $\mathbf{S}\hat{\mathbf{v}}=\mathbf{0}$ and the objective is $\mathbf{c}^{\top}\hat{\mathbf{v}}$, where $\mathbf{c}\in\mathbb{R}^{|\mathcal{R}|}$ is a vector that collects the objective coefficients $c_j$. A __feasible__ flux distribution satisfies the balances and bounds. The objective selects an __optimal__ distribution from the set of feasible distributions. However, the optimal solution may __not be unique__.

Let's consider an example to illustrate the FBA problem and its solution.

> __Example:__
>
> [▶ Calculate fluxes in a urea-cycle model](CHEME-5800-L6a-Example-UreaCycle-FluxBalance-Fall-2026.ipynb). In this example, we will build a stoichiometric model of the urea cycle in Human cells, estimate reaction directions and enzyme capacities to establish the flux bounds, and solve a linear program to maximize urea export. While this example is specific to the urea cycle, the same approach can be applied to any metabolic network. Further, it demonstrates the integrative nature of flux balance analysis. 

___

## Summary
We formulated flux balance analysis as a linear program, combining a metabolic reaction network, biological constraints, and a production objective.

> __Key Takeaways:__
>
> * __Stoichiometric matrix:__ The stoichiometric matrix is a mathematical representation of the metabolic network, where each row corresponds to a metabolite and each column corresponds to a reaction. The entries are the net stoichiometric coefficients. We can construct the stoichiometric matrix from a metabolic reconstruction, which is a curated set of reactions, metabolites, and stoichiometric coefficients.
>
> * __Flux balance analysis:__ Flux balance analysis is a linear programming problem that estimates the unknown fluxes in a metabolic network. The objective function represents the production of a desired compound, and the constraints are given by the stoichiometric matrix and the flux bounds. The solution to the FBA problem gives us an optimal flux distribution that maximizes the objective function while satisfying the constraints.
>
> * __Flux bounds:__ The flux bounds are constraints on the unknown fluxes that define the feasible solution space. They are based on reaction reversibility, enzyme capacity, and other biological factors. The bounds can be modeled as a function of the maximum reaction velocity, the catalytic constant, and the characteristic enzyme abundance. Thus, they integrate many types of genetic and biochemical information into the FBA problem.

Flux balance analysis is a powerful tool for metabolic engineering, allowing us to predict how changes in the metabolic network can affect the production of desired compounds. By understanding the stoichiometric matrix, formulating the FBA problem, and developing flux bounds, we can gain insights into the metabolic capabilities of cells and design strategies for optimizing production.

___